# 基础概念与基础掌握
`AnnData`是专门为[[#类矩阵数据]]所设计的。这意味着我们有 𝑛 个观测值，每个观测值都可以表示为 𝑑 维向量，其中每个维度对应一个变量或特征。这个 𝑛 ×𝑑 矩阵的行和列都很特殊，因为它们都带有索引。
>[!tips]
>观测值我们可以看作是唯一的一个细胞即一个细胞即为一个观测值。在一个单细胞数据中我们的一个细胞即为一个观测值而每一个细胞的变量数就是我们的特征维度变量数可以看作为基因数。

### 单细胞数据（`scRNA-seq`）举例
在scRNA-seq数据中，每一行对应一个带有条形码的细胞，每一列对应一个带有基因ID的基因。此外，对于每个细胞和每个基因，我们可能还有额外的元数据，
1. 每个细胞的供体信息
2. 每个基因的替代基因符号。
最后，我们可能还有其他非结构化元数据，比如用于绘图的调色板。
## 优点
1. Handles sparsity  处理稀疏性
2. Handles unstructured data  
    处理非结构化数据
3. Handles observation- and feature-level metadata  
    处理观测级和特征级元数据
4. s user-friendly  用户友好
   以下代码为一段引入$AnnData$依赖
   ```python
import numpy as np
import pandas as pd
import anndata as ad
from scipy.sparse import csr_matrix
from importlib.metadata import version
print(version("anndata"))
   ```
## 初始化$AnnData$
1. 首先构建一个包含一些稀疏计数信息的基本对象这些对象可能代表基因表达计数。
我们通过坡松分布创建了有100个细胞每一个细胞含有两千基因

In [11]:
counts = csr_matrix(np.random.poisson(1,size=(100,2000)),dtype=np.float32)
adata = ad.AnnData(counts)
adata
print(adata)

AnnData object with n_obs × n_vars = 100 × 2000


2. 然后我们可以通过传入的初始数据来进行访问

In [3]:
adata.X

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 125767 stored elements and shape (100, 2000)>
  Coords	Values
  (0, 0)	1.0
  (0, 1)	1.0
  (0, 2)	1.0
  (0, 3)	2.0
  (0, 4)	2.0
  (0, 5)	1.0
  (0, 6)	1.0
  (0, 7)	1.0
  (0, 8)	2.0
  (0, 9)	2.0
  (0, 10)	4.0
  (0, 11)	1.0
  (0, 13)	1.0
  (0, 14)	2.0
  (0, 15)	1.0
  (0, 16)	1.0
  (0, 17)	1.0
  (0, 19)	1.0
  (0, 21)	2.0
  (0, 22)	4.0
  (0, 23)	2.0
  (0, 24)	1.0
  (0, 25)	2.0
  (0, 28)	1.0
  (0, 30)	2.0
  :	:
  (99, 1955)	2.0
  (99, 1956)	1.0
  (99, 1959)	1.0
  (99, 1960)	1.0
  (99, 1962)	2.0
  (99, 1963)	3.0
  (99, 1968)	2.0
  (99, 1969)	2.0
  (99, 1971)	2.0
  (99, 1975)	1.0
  (99, 1980)	3.0
  (99, 1982)	1.0
  (99, 1983)	1.0
  (99, 1985)	1.0
  (99, 1986)	2.0
  (99, 1987)	2.0
  (99, 1988)	2.0
  (99, 1989)	2.0
  (99, 1990)	2.0
  (99, 1991)	1.0
  (99, 1992)	1.0
  (99, 1993)	1.0
  (99, 1997)	1.0
  (99, 1998)	1.0
  (99, 1999)	1.0


3. 现在使用`.obs_names`为`obs`和`var`轴提供索引

In [3]:
adata.obs_names = [f"Cell_{i:d}" for i in range(adata.n_obs)]
adata.var_names = [f"Gene_{i:d}" for i in range(adata.n_vars)]
print(adata.obs_names[:10])

Index(['Cell_0', 'Cell_1', 'Cell_2', 'Cell_3', 'Cell_4', 'Cell_5', 'Cell_6',
       'Cell_7', 'Cell_8', 'Cell_9'],
      dtype='object')


### 对$AnnData$进行子集化
对1号细胞和10号细胞分别索引5号基因和第1900号基因
这些索引值可用于对AnnData进行子集化，从而提供AnnData对象的[[#视图]]。我们可以想象，这对于将AnnData子集化为感兴趣的特定细胞类型或基因模块非常有用。

In [3]:
adata[["Cell_1", "Cell_10"], ["Gene_5", "Gene_1900"]]

View of AnnData object with n_obs × n_vars = 2 × 2


## 添加[[##元数据]]
### 观测/变量级别
现在我们有了对象的核心部分，接下来我们想要在观测和变量级别添加元数据。使用 anndata 实现这一点非常简单，adata.obs 和 adata.var 都是 Pandas DataFrame。

In [4]:
ct = np.random.choice(["B", "T", "Monocyte"], size=(adata.n_obs,))
adata.obs["cell_type"] = pd.Categorical(ct)
print(adata.obs)

        cell_type
Cell_0          T
Cell_1          B
Cell_2   Monocyte
Cell_3          B
Cell_4          B
...           ...
Cell_95  Monocyte
Cell_96         T
Cell_97         T
Cell_98         B
Cell_99         B

[100 rows x 1 columns]


### 非结构化元数据
AnnData 拥有 .uns，它允许存储任何非结构化元数据。这可以是任何内容，例如一个列表或字典，其中包含一些在分析我们的数据时很有用的通用信息。·

In [3]:
adata.uns["random"] = [1, 2, 3]
print(adata.uns)

OrderedDict({'random': [1, 2, 3]})


#### 层
我们可能会有原始核心数据的不同形式，例如一个是经过标准化的，另一个则没有。这些可以存储在 AnnData 的不同层中。例如，让我们对原始数据进行对数变换，并将其存储在一个层中：

In [3]:
adata.layers["log_transformed"] = np.log1p(adata.X)
print(adata)

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type'
    uns: 'random'
    layers: 'log_transformed'


# 通过 annbatch 将 PyTorch 模型与 AnnData 对象集合对接
## 引用包
>[!WARNNING]
>`pip install pyro`并不能安装这个包，请使用以下安装方式`pip install pyro-ppl`

In [4]:
import torch
import torch.nn as nn
import pyro
import pyro.distributions as dist
import numpy as np
import scanpy as sc
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from annbatch import Loader
import anndata as ad
import pooch
pyro.clear_param_store()
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, out_dim):
        super().__init__()
        
        modules = []
        for in_size, out_size in zip([input_dim]+hidden_dims, hidden_dims):
            modules.append(nn.Linear(in_size, out_size))
            modules.append(nn.LayerNorm(out_size))
            modules.append(nn.ReLU())
            modules.append(nn.Dropout(p=0.05))
        modules.append(nn.Linear(hidden_dims[-1], out_dim))
        self.fc = nn.Sequential(*modules)
    
    def forward(self, *inputs):
        shape = dist.util.broadcast_shape(*[s.shape[:-1] for s in inputs]) + (-1,)
        inputs = [s.expand(shape) for s in inputs]
        
        input_cat = torch.cat(inputs, dim=-1)
        return self.fc(input_cat)
class CVAE(nn.Module):
    def __init__(self, input_dim, n_conds, n_classes, hidden_dims, latent_dim, classifier_dims=[128]):
        super().__init__()
        
        self.encoder = MLP(input_dim+n_conds, hidden_dims, 2*latent_dim) # output - mean and logvar of z
        
        self.decoder = MLP(latent_dim+n_conds+n_classes, hidden_dims[::-1], input_dim)
        self.theta = nn.Linear(n_conds, input_dim, bias=False)
        
        self.classifier = MLP(latent_dim, classifier_dims, n_classes)
        
        self.latent_dim = latent_dim
    
    def model(self, x, batches, classes, size_factors, supervised):
        pyro.module("cvae", self)
        
        batch_size = x.shape[0]
        
        with pyro.plate("data", batch_size):
            z_loc = x.new_zeros((batch_size, self.latent_dim))
            z_scale = x.new_ones((batch_size, self.latent_dim))
            z = pyro.sample("Z", dist.Normal(z_loc, z_scale).to_event(1))
            
            classes_probs = self.classifier(z).softmax(dim=-1)
            if supervised:
                obs = classes
            else:
                obs = None
            classes = pyro.sample("Class", dist.OneHotCategorical(probs=classes_probs), obs=obs)
            
            dec_mu = self.decoder(z, batches, classes).softmax(dim=-1) * size_factors[:, None]
            dec_theta = torch.exp(self.theta(batches))
            
            logits = (dec_mu + 1e-6).log() - (dec_theta + 1e-6).log()
            
            pyro.sample("X", dist.NegativeBinomial(total_count=dec_theta, logits=logits).to_event(1), obs=x.int())
        
    def guide(self, x, batches, classes, size_factors, supervised):
        batch_size = x.shape[0]
        
        with pyro.plate("data", batch_size):
            z_loc_scale = self.encoder(x, batches)
            
            z_mu = z_loc_scale[:, :self.latent_dim]
            z_var = torch.sqrt(torch.exp(z_loc_scale[:, self.latent_dim:]) + 1e-4)
            
            z = pyro.sample("Z", dist.Normal(z_mu, z_var).to_event(1))
            
            if not supervised:
                classes_probs = self.classifier(z).softmax(dim=-1)
                pyro.sample("Class", dist.OneHotCategorical(probs=classes_probs))

## 从两个$AnnData$对象拼接一个出来

In [2]:
def download(url, fname, sha_hash):
    pooch.retrieve(
        url=url,
        known_hash=sha_hash,
        fname=fname,
        path=".",
        downloader=pooch.HTTPDownloader(
            progressbar=True,
            chunk_size=1024,
            timeout=120,
            headers={"User-Agent": "an/ndata/1.0.0 (https://github.com/scverse/anndata)"},
        ),
    )
download("https://scverse-exampledata.s3.eu-west-1.amazonaws.com/anndata/cite_covid_full.h5ad", "covid_cite.h5ad", "cb9745b2d642459926194961f34110a17c16ef6b11777304a3478e12bd682657")
download("https://scverse-exampledata.s3.eu-west-1.amazonaws.com/anndata/pbmc_seurat_v4.h5ad", "pbmc_seurat_v4.h5ad", "c3b0100a6ce27beb64eff53692e09f98da2a58cfdfea08d15ff204f834b41396")

100%|########################################| 304M/304M [00:00<00:00, 844GB/s]
 39%|###############                       | 396M/1.00G [04:08<05:27, 1.85MB/s]Traceback (most recent call last):
  File "D:\data\rlcao\.jupymd\Lib\site-packages\urllib3\response.py", line 905, in _error_catcher
    yield
  File "D:\data\rlcao\.jupymd\Lib\site-packages\urllib3\response.py", line 1043, in _raw_read
    raise IncompleteRead(self._fp_bytes_read, self.length_remaining)
urllib3.exceptions.IncompleteRead: IncompleteRead(395967692 bytes read, 606611124 more expected)

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "D:\data\rlcao\.jupymd\Lib\site-packages\requests\models.py", line 937, in generate
    yield from self.raw.stream(chunk_size, decode_content=True)
  File "D:\data\rlcao\.jupymd\Lib\site-packages\urllib3\response.py", line 1267, in stream
    data = self.read(amt=amt, decode_content=decode_content)
  File "D:\data\rlcao\.ju

# 专业术语
## 元数据
原数据METDATA就是对原有数据的一个补充和注释有了原数据数据才具备了可追溯可理解的背景。
### 元数据包含
1. 数据的来源与下载记录
   - 必须记录项目目录中所有数据的来源
   - 还需要记录数据下载的时间因为数据库的数据是会更新的
2. 样本与实验信息
3. 软件与运行环境信息
### 元数据的存储
1. 观测级与特征级元数据
2. 多维元数据
3. 非结构化元数据
### 元数据的意义
- 数据可追溯性
- 正确解释数据
- 确保可重复性
## 视图
试图就是给原始数据加了一个放大镜或者望远镜当我们透过这个镜子看数据时只可以看到数据的一部分呃但你看到的依然是原版数据本身而不是复印出来的副本因为是直接看原版数据所以随着你的镜子修改了看到的内容原本书写会跟着改变反之别人直接修改了原版数据你镜子里看到也会改变。
### 原理
仕途之所以不是不能复制原始数据是因为它仅仅改变了不长属性来重新映射索引到内存的偏移量
## 类矩阵数据
### 基础定义
- 数组：将具有相同类型的若干变量按有序形式组织起来的同类数据元素**集合**。所以我们的数组本质就是一个集合
- 矩阵：矩阵是一种变换或映射运算符的体现它的运算具有严格的数学规则。矩阵存储有规则的数据
>[!note]
>矩阵以数组的形式存在，一维数相当于向量，二维数组相当于矩阵，可将矩阵视为数组的子集

### 编程中的`类数组对象`
在编程语言中我们的类矩阵数据常常体现为类数组数据我们可以使用python列表等创建数组来数组作为矩阵